# ABR公式CSV直接照合 PoC（住所→座標）

このNotebookは、**公式アドレス・ベース・レジストリ（ABR）のCSVを直接利用し、糸満市の地番住所から
緯度経度を取得できるか確認するPoC**です。

- **sheltermatch本体（`sheltermatch.ipynb`）ではありません。**
- Jageocoder等の汎用ジオコーダーは使いません（`experiments/jageocoder_poc.ipynb`とは別物です）。
- `abrdb`/`abrg`（公式ABR Geocoder。`experiments/abr_geocoder_poc.ipynb`）も使いません。
  PostgreSQL・DuckDBも使いません。
- ABR公式データは**利用者が手動でダウンロードしてアップロード**したものだけを使います。
  Notebookから外部サイトへは一切アクセスしません。
- **現段階では地番住所のみを対象とします。** 住居表示住所（`rsdt_addr_flg=1`の町字）は、
  今回の3種類のCSVだけでは正しく座標化できるとは限らないため、無理に座標を確定せず
  `residential_display_area`として区別します。別データ（住居表示・街区/住居表示・住居等）が
  必要になる可能性があります。

## 使う公式ZIP（利用者が手動アップロード）

1. `mt_town_all.csv.zip`（全国町字マスタ）
2. `mt_parcel_city472107.csv.zip`（糸満市 地番マスタ）
3. `mt_parcel_pos_city472107.csv.zip`（糸満市 地番マスタ位置参照拡張）

ファイル名ではなく、ZIP内CSVのヘッダー列から種別を自動判定します。`abr_post_code.zip`
（郵便番号データ）が混ざっていても、今回は使わない旨を表示して無視します。


In [ ]:
# ===== 設定 =====

LG_CODE = "472107"
PREF_NAME = "沖縄県"
CITY_NAME = "糸満市"

# 個人情報は使用しない。公開されている公共施設の住所（他のPoC Notebookと同じもの）を使う。
TEST_ADDRESSES = [
    "沖縄県糸満市潮崎町1丁目1番地",  # 糸満市役所
    "沖縄県糸満市字糸満673",  # 糸満市立糸満小学校
    "沖縄県糸満市真栄里1448番地",  # 糸満市立中央図書館
]

print("設定を読み込みました。")
print(f"  LG_CODE   = '{LG_CODE}' ({PREF_NAME}{CITY_NAME})")


In [ ]:
# ===== ABRデータ読込・抽出・マスター生成 =====
# ZIPアップロード→CSV種別判定→糸満市データ抽出→地番＋位置参照＋町字マスタの結合までの、
# 一連の初期化処理をまとめて行う。

# --- ABR ZIPアップロード ---
# mt_town_all.csv.zip / mt_parcel_city472107.csv.zip / mt_parcel_pos_city472107.csv.zip を
# まとめてアップロードする。ファイル名ではなく、ZIP内CSVのヘッダー列から種別を自動判定する。
# 不足しているデータを外部サイトから自動取得することはしない。

import io
import json
import re
import unicodedata
import zipfile

import pandas as pd
from google.colab import files

# 種別判定に使う識別列（公式ABR CSVの列構成に基づく。列が全部揃っているものだけを該当種別とみなす）。
# 地番位置参照(rep_lon/rep_lat)→地番(prc_num1-3)→町字 の順で判定する
# （地番位置参照側の列が地番側の列も含んでいる可能性があるため、より限定的な方を先に見る）。
PARCEL_POS_SIGNATURE = {"lg_code", "machiaza_id", "prc_id", "rep_lon", "rep_lat"}
PARCEL_SIGNATURE = {"lg_code", "machiaza_id", "prc_id", "prc_num1", "prc_num2", "prc_num3"}
TOWN_SIGNATURE = {"lg_code", "machiaza_id", "oaza_cho", "chome", "koaza", "machiaza_dist", "rsdt_addr_flg"}


def classify_csv(columns):
    cols = set(columns)
    if PARCEL_POS_SIGNATURE.issubset(cols):
        return "parcel_pos"
    if PARCEL_SIGNATURE.issubset(cols):
        return "parcel"
    if TOWN_SIGNATURE.issubset(cols):
        return "town"
    if "post_code" in cols:
        return "post_code"
    return None


def read_abr_csv(fileobj):
    """ABR CSVを読み込む。ID・番号列の先頭ゼロや桁落ちを防ぐため全列を文字列として読み込み、
    欠損値は独自にNaN化せず空文字列のまま扱う（文字列連結・突合を単純にするため）。"""
    raw = fileobj.read()
    last_error = None
    for encoding in ("utf-8-sig", "cp932", "utf-8"):
        try:
            return pd.read_csv(
                io.BytesIO(raw), dtype=str, keep_default_na=False, na_values=[], encoding=encoding
            )
        except (UnicodeDecodeError, UnicodeError) as e:
            last_error = e
            continue
    raise ValueError(f"文字コードを判定できなかった: {last_error}")


print("mt_town_all.csv.zip / mt_parcel_city472107.csv.zip / mt_parcel_pos_city472107.csv.zip を")
print("まとめて選択してください。")
uploaded_zips = files.upload()

town_df = None
parcel_df = None
parcel_pos_df = None

for zip_filename, zip_bytes in uploaded_zips.items():
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            print(f"'{zip_filename}' 内にCSVが見つからないため無視する。")
            continue
        for member in csv_members:
            with zf.open(member) as f:
                df = read_abr_csv(f)
            kind = classify_csv(df.columns)
            if kind == "town":
                print(f"'{zip_filename}' 内 '{member}' を町字マスタとして認識した。")
                town_df = df
            elif kind == "parcel":
                print(f"'{zip_filename}' 内 '{member}' を地番マスタとして認識した。")
                parcel_df = df
            elif kind == "parcel_pos":
                print(f"'{zip_filename}' 内 '{member}' を地番位置参照として認識した。")
                parcel_pos_df = df
            elif kind == "post_code":
                print(f"'{zip_filename}' 内 '{member}' は郵便番号データのため、今回の変換には使用しない。")
            else:
                print(f"'{zip_filename}' 内 '{member}' は既知の列構成と一致しないため無視する。")

print()
print(f"町字マスタ    : {'OK' if town_df is not None else '不足'}")
print(f"地番マスタ    : {'OK' if parcel_df is not None else '不足'}")
print(f"地番位置参照  : {'OK' if parcel_pos_df is not None else '不足'}")

poc_status = {}
poc_status["zip_load"] = all(df is not None for df in (town_df, parcel_df, parcel_pos_df))

# --- 糸満市データ抽出 ---
# 町字マスタは全国データなのでlg_codeで抽出する。地番・位置参照も同じ自治体コードであることを確認する。

if not poc_status.get("zip_load"):
    print("必要なABR CSVが揃っていないため、このセルはスキップする。")
    poc_status["extract"] = False
else:
    town_lg = town_df[town_df["lg_code"] == LG_CODE].copy()
    parcel_lg = parcel_df[parcel_df["lg_code"] == LG_CODE].copy()
    parcel_pos_lg = parcel_pos_df[parcel_pos_df["lg_code"] == LG_CODE].copy()

    other_lg_in_parcel = (parcel_df["lg_code"] != LG_CODE).sum()
    other_lg_in_pos = (parcel_pos_df["lg_code"] != LG_CODE).sum()
    if other_lg_in_parcel:
        print(f"注意: 地番マスタに{LG_CODE}以外のlg_codeが{other_lg_in_parcel}件含まれている。")
    if other_lg_in_pos:
        print(f"注意: 地番位置参照に{LG_CODE}以外のlg_codeが{other_lg_in_pos}件含まれている。")

    print(f"{CITY_NAME}町字件数: {len(town_lg)}件")
    print(f"地番件数: {len(parcel_lg)}件")
    print(f"地番位置参照件数: {len(parcel_pos_lg)}件")

    print("\nrsdt_addr_flgの内訳（ABR上の住居表示フラグ。値の意味は独自解釈しない）:")
    print(town_lg["rsdt_addr_flg"].value_counts(dropna=False).to_string())

    town_lg["town_label"] = (
        town_lg["oaza_cho"] + town_lg["chome"] + town_lg["koaza"] + town_lg["machiaza_dist"]
    )

    residential_towns = town_lg[town_lg["rsdt_addr_flg"] == "1"]
    if len(residential_towns):
        print(f"\nrsdt_addr_flg=1（住居表示）の町字: {len(residential_towns)}件")
        for label in residential_towns["town_label"]:
            print(f"  - {label}")
    else:
        print("\nrsdt_addr_flg=1（住居表示）の町字はない。")

    poc_status["extract"] = len(town_lg) > 0 and len(parcel_lg) > 0


# --- 地番＋座標マスター生成 ---
# 地番マスタ×地番位置参照(lg_code, machiaza_id, prc_id)、その結果×町字マスタ(lg_code, machiaza_id)で
# 結合する。キー重複がある場合はdrop_duplicates()で勝手に1件へ潰さず、validate=で結合関係を確認する。

if not poc_status.get("extract"):
    print("糸満市データの抽出が完了していないため、このセルはスキップする。")
    poc_status["merge"] = False
else:
    parcel_dupe = int(parcel_lg.duplicated(subset=["lg_code", "machiaza_id", "prc_id"]).sum())
    pos_dupe = int(parcel_pos_lg.duplicated(subset=["lg_code", "machiaza_id", "prc_id"]).sum())
    town_dupe = int(town_lg.duplicated(subset=["lg_code", "machiaza_id"]).sum())
    print(f"地番キー重複: {parcel_dupe}件")
    print(f"位置参照キー重複: {pos_dupe}件")
    print(f"町字キー重複: {town_dupe}件")

    try:
        parcel_master = parcel_lg.merge(
            parcel_pos_lg[["lg_code", "machiaza_id", "prc_id", "rep_lon", "rep_lat"]],
            on=["lg_code", "machiaza_id", "prc_id"],
            how="left",
            validate="one_to_one",
        )
        parcel_master = parcel_master.merge(
            town_lg[["lg_code", "machiaza_id", "town_label", "rsdt_addr_flg"]],
            on=["lg_code", "machiaza_id"],
            how="left",
            validate="many_to_one",
        )
        merge_ok = True
    except Exception as e:
        print(f"結合に失敗した（キー重複等の可能性がある）: {type(e).__name__}: {e}")
        parcel_master = None
        merge_ok = False

    poc_status["merge"] = merge_ok
    if merge_ok:
        parcel_master["rep_lon"] = pd.to_numeric(parcel_master["rep_lon"], errors="coerce")
        parcel_master["rep_lat"] = pd.to_numeric(parcel_master["rep_lat"], errors="coerce")
        has_coords = parcel_master["rep_lon"].notna() & parcel_master["rep_lat"].notna()
        print(f"\n全地番件数: {len(parcel_master)}件")
        print(f"座標あり件数: {int(has_coords.sum())}件")
        print(f"座標なし件数: {int((~has_coords).sum())}件")


In [ ]:
# ===== 住所変換ロジック定義 =====
# 正規化→町字インデックス作成・照合→地番解析・照合→geocode_one_address() までの、
# 住所→座標変換ロジックをまとめて定義する。

# --- 住所正規化関数 ---
# PoCとして必要十分な範囲のみ正規化する。住所正規化ライブラリの新規導入や、
# 巨大な独自住所パーサーの実装はしない。

HYPHEN_CHARS = "－ー‐‑–—―─−"
HYPHEN_TRANS = str.maketrans({c: "-" for c in HYPHEN_CHARS})


def normalize_address(text):
    """Unicode NFKC正規化（全角数字→半角等）、ハイフン類の統一、空白除去を行う。"""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(HYPHEN_TRANS)
    text = re.sub(r"\s+", "", text)
    return text


def strip_pref_city(text):
    """先頭の都道府県名・市区町村名を取り除く（無くても動くようにする）。"""
    for prefix in (PREF_NAME + CITY_NAME, CITY_NAME, PREF_NAME):
        if text.startswith(prefix):
            return text[len(prefix):]
    return text


print("normalize_address() / strip_pref_city() を定義した。")
print("例:", strip_pref_city(normalize_address(" 沖縄県糸満市字糸満６７３ ")))


# --- 町字判定 ---
# oaza_cho+chome+koaza+machiaza_distで町字表記を作り、入力住所の先頭と最も長く一致する町字を採用する。
# ABR原文は全角数字（例:「１丁目」）を含み得るため、検索キーには入力住所と同じnormalize_address()を
# 適用する（表示用のofficial_labelはABR原文のまま保持する）。沖縄県でよくある「字」の有無は、
# 正式名称は保持したまま、検索用の別名として先頭の「字」を省略した形も許容する。
# 同じ長さで複数machiaza_idに一致する場合はambiguous_townとする。


def build_town_index(town_lg):
    """町字マスタから検索用インデックスを作る。ABR原文の表記（全角数字等を含み得る）を
    official_labelとしてそのまま保持しつつ、search_keyは入力住所と同じnormalize_address()を
    通して作る（入力側だけを正規化して原文のままの町字と比較すると、全角丁目表記等で
    一致しなくなるため）。"""
    entries = []
    for _, row in town_lg.iterrows():
        label = row["town_label"]
        if not label:
            continue
        search_key = normalize_address(label)
        if not search_key:
            continue
        entries.append(
            {"search_key": search_key, "official_label": label, "machiaza_id": row["machiaza_id"],
             "rsdt_addr_flg": row["rsdt_addr_flg"]}
        )
        if search_key.startswith("字") and len(search_key) > 1:
            alias = search_key[1:]
            entries.append(
                {"search_key": alias, "official_label": label, "machiaza_id": row["machiaza_id"],
                 "rsdt_addr_flg": row["rsdt_addr_flg"]}
            )
    return entries


def match_town(remainder, town_index):
    """remainderの先頭と最も長く一致する町字を返す。
    戻り値は (一致したエントリ or None, 一致部分を除いた残り文字列, 曖昧かどうか)。"""
    best_len = -1
    candidates = []
    for entry in town_index:
        key = entry["search_key"]
        if key and remainder.startswith(key):
            if len(key) > best_len:
                best_len = len(key)
                candidates = [entry]
            elif len(key) == best_len:
                candidates.append(entry)

    if not candidates:
        return None, remainder, False

    distinct_ids = {c["machiaza_id"] for c in candidates}
    if len(distinct_ids) > 1:
        return None, remainder, True

    return candidates[0], remainder[best_len:], False


if poc_status.get("extract"):
    TOWN_INDEX = build_town_index(town_lg)
    print(f"町字インデックスを{len(TOWN_INDEX)}件（別名込み）作成した。")
else:
    TOWN_INDEX = []
    print("糸満市データの抽出が完了していないため、町字インデックスは作成しない。")


# --- 地番判定 ---
# 町字確定後の残り文字列からprc_num1/2/3を取得し、地番マスタと(lg_code, machiaza_id,
# prc_num1, prc_num2, prc_num3)で照合する。前方一致だけで座標を決定しない
# （例: "673" と "673-2" は別地番として扱う）。候補が複数残る場合はambiguous_parcelとする。


def parse_parcel_numbers(text):
    """'673' '673番地' '673-2' '673番地の2' 等からprc_num1〜3を抽出する。
    数字として解釈できない場合や、4つ以上の要素に分かれる場合はNoneを返す
    （prc_num1〜3の3要素までしか無いため、黙って切り詰めて別地番へ誤一致させない）。"""
    if not text:
        return None
    t = text.strip()
    t = re.sub(r"番地の|番の", "-", t)
    t = re.sub(r"番地|番|号", "", t)
    t = t.strip("-").strip()
    if not t:
        return None

    parts = [p for p in re.split(r"-+", t) if p != ""]
    if not parts or len(parts) > 3 or any(not p.isdigit() for p in parts):
        return None

    parts = (parts + ["", ""])[:3]
    return tuple(parts)


def match_parcel(machiaza_id, numbers, parcel_master):
    n1, n2, n3 = numbers
    subset = parcel_master[parcel_master["machiaza_id"] == machiaza_id]
    subset = subset[
        (subset["prc_num1"] == n1) & (subset["prc_num2"] == n2) & (subset["prc_num3"] == n3)
    ]
    return subset


print("parse_parcel_numbers() / match_parcel() を定義した。")
print("例:", parse_parcel_numbers("673番地の2"))


# --- 住居表示地域の扱い・geocode_one_address() ---
# 町字のrsdt_addr_flg==1の場合、今回の地番データだけで住居表示住所を正しく座標化できるとは限らない。
# 「地番マスタで似た番号が見つかったから正しい」とは扱わず、地番として無理に確定せず
# residential_display_areaの状態を返す。ここまでの正規化・町字判定・地番判定を組み合わせる中心処理になる。


def _result(input_address, status, normalized_address=None, matched_town=None, machiaza_id=None,
            parcel_number=None, prc_id=None, longitude=None, latitude=None, rsdt_addr_flg=None):
    return {
        "input_address": input_address,
        "normalized_address": normalized_address,
        "matched_town": matched_town,
        "machiaza_id": machiaza_id,
        "parcel_number": parcel_number,
        "prc_id": prc_id,
        "longitude": longitude,
        "latitude": latitude,
        "rsdt_addr_flg": rsdt_addr_flg,
        "status": status,
    }


def geocode_one_address(raw_address):
    if pd.isna(raw_address) or not str(raw_address).strip():
        return _result(raw_address, "blank_address")

    normalized = normalize_address(raw_address)
    remainder = strip_pref_city(normalized)

    town_entry, town_remainder, town_ambiguous = match_town(remainder, TOWN_INDEX)
    if town_ambiguous:
        return _result(raw_address, "ambiguous_town", normalized_address=normalized)
    if town_entry is None:
        return _result(raw_address, "town_not_found", normalized_address=normalized)

    common_kwargs = dict(
        normalized_address=normalized,
        matched_town=town_entry["official_label"],
        machiaza_id=town_entry["machiaza_id"],
        rsdt_addr_flg=town_entry["rsdt_addr_flg"],
    )

    # 住居表示地域は、今回の地番データだけでは正しく座標化できるとは限らないため、
    # 地番判定を試みる前にここで確定させる（偶然一致した地番があっても採用しない）。
    if town_entry["rsdt_addr_flg"] == "1":
        return _result(raw_address, "residential_display_area", **common_kwargs)

    numbers = parse_parcel_numbers(town_remainder)
    if numbers is None:
        return _result(raw_address, "parcel_not_found", **common_kwargs)

    parcel_number_display = "-".join(p for p in numbers if p)
    matches = match_parcel(town_entry["machiaza_id"], numbers, PARCEL_MASTER)

    if len(matches) == 0:
        return _result(raw_address, "parcel_not_found", parcel_number=parcel_number_display, **common_kwargs)
    if len(matches) > 1:
        return _result(raw_address, "ambiguous_parcel", parcel_number=parcel_number_display, **common_kwargs)

    row = matches.iloc[0]
    if pd.isna(row["rep_lon"]) or pd.isna(row["rep_lat"]):
        return _result(
            raw_address, "coordinates_missing", parcel_number=parcel_number_display,
            prc_id=row["prc_id"], **common_kwargs
        )

    return _result(
        raw_address, "matched", parcel_number=parcel_number_display, prc_id=row["prc_id"],
        longitude=row["rep_lon"], latitude=row["rep_lat"], **common_kwargs
    )


if poc_status.get("merge"):
    PARCEL_MASTER = parcel_master
    print("geocode_one_address() を定義した（町字判定→住居表示チェック→地番判定の順で処理する）。")
else:
    PARCEL_MASTER = None
    print("地番・位置参照の結合が完了していないため、geocode_one_address()は定義のみ行う。")


In [ ]:
# ===== 公開住所3件で実行 =====
# 座標値はコードへハードコードせず、アップロードしたABRデータから取得する。

if not poc_status.get("merge"):
    print("地番・位置参照の結合が完了していないため、このセルはスキップする。")
    poc_status["public_test"] = None
else:
    test_results = [geocode_one_address(addr) for addr in TEST_ADDRESSES]
    matched_count = sum(1 for r in test_results if r["status"] == "matched")
    print(f"{len(TEST_ADDRESSES)}件中{matched_count}件が座標まで一致した。")
    poc_status["public_test"] = matched_count
    display(pd.DataFrame(test_results))


In [ ]:
# ===== 任意CSV一括変換（検証用） =====
# address列を持つCSVをアップロードすると一括変換を試せる。address列以外の列名には依存せず、
# 元の列を保持したまま変換結果列を追加する。変換できない行も削除しない。
# sheltermatch本体のCSV仕様への統合はここでは行わない。

if not poc_status.get("merge"):
    print("地番・位置参照の結合が完了していないため、このセルはスキップする。")
    poc_status["csv_batch"] = None
else:
    print("address列を持つCSVを選択してください。その他の列はそのまま保持します。")
    print("（試さない場合はアップロードをキャンセルしてください）")
    uploaded_csv = files.upload()

    if not uploaded_csv:
        print("CSVがアップロードされなかったため、このセルはスキップされた。")
        poc_status["csv_batch"] = None
    else:
        csv_filename = list(uploaded_csv.keys())[0]
        addresses_df = pd.read_csv(io.BytesIO(uploaded_csv[csv_filename]))

        if "address" not in addresses_df.columns:
            print("'address'列が見つかりません。住所を格納したaddress列を用意してください。")
            poc_status["csv_batch"] = False
        else:
            batch_results = [geocode_one_address(addr) for addr in addresses_df["address"]]
            batch_df = pd.DataFrame(batch_results)

            addresses_df["matched_town"] = batch_df["matched_town"]
            addresses_df["machiaza_id"] = batch_df["machiaza_id"]
            addresses_df["prc_id"] = batch_df["prc_id"]
            addresses_df["latitude"] = batch_df["latitude"]
            addresses_df["longitude"] = batch_df["longitude"]
            addresses_df["geocode_status"] = batch_df["status"]

            matched_count = int((addresses_df["geocode_status"] == "matched").sum())
            print(f"{len(addresses_df)}行中{matched_count}行が座標まで一致した。")
            display(addresses_df)

            print("\ngeocode_status内訳:")
            print(addresses_df["geocode_status"].value_counts().to_string())

            poc_status["csv_batch"] = matched_count > 0


In [ ]:
# ===== 検証結果サマリ =====


def _fmt(value):
    if value is True:
        return "OK"
    if value is False:
        return "NG"
    return "未実施"


print("ABR公式CSV直接照合PoC 検証結果サマリ")
print(f"  ABR ZIP読込          : {_fmt(poc_status.get('zip_load'))}")
print(f"  {CITY_NAME}町字抽出        : {_fmt(poc_status.get('extract'))}")
print(f"  地番・位置参照結合    : {_fmt(poc_status.get('merge'))}")

public_test = poc_status.get("public_test")
if public_test is None:
    print("  公開住所テスト        : 未実施")
else:
    print(f"  公開住所テスト        : {public_test}/{len(TEST_ADDRESSES)}件")

print(f"  CSV一括変換           : {_fmt(poc_status.get('csv_batch'))}")

if poc_status.get("merge"):
    residential_count = int((town_lg["rsdt_addr_flg"] == "1").sum())
    coords_missing_count = int(
        (~(parcel_master["rep_lon"].notna() & parcel_master["rep_lat"].notna())).sum()
    )
    print(f"  住居表示地域          : {residential_count}町字")
    print(f"  座標なし地番          : {coords_missing_count}件")
else:
    residential_count = None

print()
if public_test:
    print(
        "地番住所については、公式ABR CSV（町字・地番・地番位置参照）の直接照合だけで"
        "座標取得できる可能性がある。次の検証（件数を増やす・実運用データでの確認等）へ進める材料がある。"
    )
elif public_test == 0:
    print("公開住所テストで座標まで一致した件数が0件だった。町字・地番の判定結果を確認すること。")
else:
    print("公開住所テストが未実施のため、この時点では判断できない。")

if residential_count:
    print(
        f"{CITY_NAME}には住居表示地域が{residential_count}町字あるため、"
        "それらの住所を扱うには住居表示・街区/住居表示・住居データの追加が必要になる可能性がある。"
    )

print()
print("この結果は本Notebook内の検証にとどまり、sheltermatch本体へは反映していません。")
